# LOOM - Phase 1: QLoRA fine-tune on T4 (16 GB)

Teaches DeepSeek-R1-Distill-Qwen-7B the two-phase trace shape so a clean concept-naming
moment exists to hook in Phase 2.

**Runtime > Change runtime type > T4 GPU** before running.

Everything runs in bitsandbytes 4-bit. Important: 4-bit quantizes **weights only** - the
residual stream still flows in fp16, so the activation vectors you extract in Phase 2 are
genuine full-precision hidden states. The module tree also stays standard
(`model.model.layers[i]`), so `register_forward_hook` works normally. This is exactly why we
are not using the AWQ build.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate peft bitsandbytes datasets

import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print('gpu:', torch.cuda.get_device_name(0))
print('vram: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# Native bf16 needs compute capability >= 8.0 (Ampere). T4 is sm_75, so we use fp16.
# NB: do NOT use torch.cuda.is_bf16_supported() to detect this - on recent torch it returns
# True on a T4 because it counts *emulated* bf16.
cc = torch.cuda.get_device_capability(0)
NATIVE_BF16 = cc[0] >= 8
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
print(f'compute capability {cc[0]}.{cc[1]} | native bf16 {NATIVE_BF16} -> training in {"bf16" if NATIVE_BF16 else "fp16"}')

## 1. Dataset

Upload `train.jsonl` (72 rows) from `compact discussion/mechanics/`.

In [ ]:
import json, pathlib, shutil
from google.colab import drive

# force_remount: interrupting a previous mount leaves /content/drive present but empty,
# which makes the file look missing. Always remount rather than trusting the folder exists.
drive.mount('/content/drive', force_remount=True)

root = pathlib.Path('/content/drive/MyDrive')
print('MyDrive entries:', len(list(root.iterdir())))   # 0 here means the mount failed

src = root / 'train.jsonl'
if not src.exists():
    print('train.jsonl not in Drive root. .jsonl files found:')
    for p in root.glob('*.jsonl'):
        print('  ', p.name)
    raise FileNotFoundError(src)

shutil.copy(src, 'train.jsonl')
print('copied %.0f KB' % (pathlib.Path('train.jsonl').stat().st_size / 1024))

rows = [json.loads(l) for l in open('train.jsonl', encoding='utf-8')]
train_rows = [r for r in rows if r['split'] == 'train']
test_rows  = [r for r in rows if r['split'] == 'test']

print(f'total {len(rows)} | train {len(train_rows)} | held-out {len(test_rows)}')
print('full :', sum(r['form'] == 'full'  for r in train_rows))
print('short:', sum(r['form'] == 'short' for r in train_rows))
assert len(train_rows) == 52 and len(test_rows) == 20

## 2. Load model in 4-bit

~4.5 GB of weights, leaving plenty of T4 headroom. `compute_dtype=float16` because Turing
has no bf16.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# transformers v5 renamed the `torch_dtype` argument to `dtype`
_dtype_kw = 'dtype' if int(transformers.__version__.split('.')[0]) >= 5 else 'torch_dtype'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={'': 0}, **{_dtype_kw: DTYPE},
)
model.config.use_cache = False

n_layers = model.config.num_hidden_layers
print(f'layers {n_layers} | hidden {model.config.hidden_size}')
print('vram used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

## 3. Hook smoke test (run this BEFORE training)

Phase 2 lives or dies on being able to read and write the residual stream. Two minutes here
saves you discovering a broken hook after a training run. We confirm the read hook fires and
returns fp16 hidden states, then confirm a write hook can actually perturb the output.

In [ ]:
LAYER = 16  # middle third of 28 - sweep this properly in Phase 2

captured = {}
def read_hook(mod, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    captured['h'] = h.detach()

probe = tok('A car brakes from 24 m/s at 9 m/s^2. How far does it skid?', return_tensors='pt').to(0)

handle = model.model.layers[LAYER].register_forward_hook(read_hook)
with torch.no_grad():
    model(**probe)
handle.remove()

h = captured['h']
print(f'READ  ok | shape {tuple(h.shape)} | dtype {h.dtype} | norm {h[0, -1].norm().item():.1f}')
assert h.shape[-1] == model.config.hidden_size, 'hidden size mismatch - wrong module hooked'
assert h.dtype == DTYPE, f'expected {DTYPE} activations, got {h.dtype}'

# write test: add a scaled random vector, confirm the logits actually move
v = torch.randn(model.config.hidden_size, dtype=DTYPE, device=0)
v = v / v.norm()

def write_hook(mod, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    h[:, -1, :] += 0.5 * v * h[:, -1, :].norm()   # norm-matched injection
    return (h,) + out[1:] if isinstance(out, tuple) else h

with torch.no_grad():
    base = model(**probe).logits[0, -1].float()
handle = model.model.layers[LAYER].register_forward_hook(write_hook)
with torch.no_grad():
    moved = model(**probe).logits[0, -1].float()
handle.remove()

delta = (moved - base).abs().max().item()
print(f'WRITE ok | max logit shift {delta:.3f}')
assert delta > 1e-3, 'write hook had no effect - injection would silently do nothing'
print('\nboth hooks functional -> Phase 2 is viable on this setup')

## 4. Format traces

We drop the `[PROMPT]` / `[REASONING]` / `[SOLUTION]` scaffolding and use the model's own
chat template, keeping only `<think>...</think>` - which R1-Distill emits natively. Aligning
with the native tags means less fighting during training, and makes the Phase 3 metric
(tokens between `<think>` and `</think>`) directly comparable to the untuned baseline.

Loss is masked over the prompt so the model learns the **response shape**, not the problems.

In [ ]:
MAX_LEN = 768

def build(row):
    prefix = tok.apply_chat_template(
        [{'role': 'user', 'content': row['prompt']}],
        tokenize=False, add_generation_prompt=True,
    )
    response = f"<think>\n{row['think']}\n</think>\n\n{row['solution']}"

    p_ids = tok(prefix, add_special_tokens=False)['input_ids']
    r_ids = tok(response, add_special_tokens=False)['input_ids'] + [tok.eos_token_id]

    ids = (p_ids + r_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + r_ids)[:MAX_LEN]
    return {'input_ids': ids, 'labels': labels}

train_ds = [build(r) for r in train_rows]

lens = [len(d['input_ids']) for d in train_ds]
print(f'token length: min {min(lens)} | mean {sum(lens)//len(lens)} | max {max(lens)}')
assert max(lens) < MAX_LEN, 'a trace is being truncated - raise MAX_LEN'

print('\n--- sample (supervised part only) ---')
d = train_ds[0]
print(tok.decode([t for t in d['labels'] if t != -100])[:400])

In [ ]:
import torch

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    ids, labels, mask = [], [], []
    for b in batch:
        pad = n - len(b['input_ids'])
        ids.append(b['input_ids'] + [tok.pad_token_id] * pad)
        labels.append(b['labels'] + [-100] * pad)
        mask.append([1] * len(b['input_ids']) + [0] * pad)
    return {
        'input_ids': torch.tensor(ids),
        'labels': torch.tensor(labels),
        'attention_mask': torch.tensor(mask),
    }

## 5. LoRA + train

52 examples is a small set, so epochs are high and LoRA rank is modest. ~100 optimizer steps,
roughly 10-20 minutes on a T4.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir='out',
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy='no',
    fp16=not NATIVE_BF16,         # T4 -> fp16
    bf16=NATIVE_BF16,             # A100/L4 -> bf16
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    report_to='none',
)

Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate).train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE = '/content/drive/MyDrive/deep_reason_lora'
model.save_pretrained(SAVE)
tok.save_pretrained(SAVE)
print('adapter saved ->', SAVE)

## 6. Baseline measurement on held-out problems

These 20 `c` variants were never trained on. Their think-token counts are the **baseline the
injection must beat** in Phase 3. Same physics as the trained `a`/`b` variants, different
surface wording - so this doubles as your cross-prompt transfer set.

In [ ]:
import re, json
model.config.use_cache = True
model.eval()

def generate(prompt, max_new=600):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(0)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

def think_tokens(text):
    m = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    return len(tok(m.group(1), add_special_tokens=False)['input_ids']) if m else None

results = []
for r in test_rows:
    gen = generate(r['prompt'])
    n = think_tokens(gen)
    results.append({'id': r['id'], 'concept_id': r['concept_id'], 'think_tokens': n, 'output': gen})
    print(f"{r['id']:10s} {r['concept_id']:32s} {n}")

ok = [x['think_tokens'] for x in results if x['think_tokens']]
print(f'\nBASELINE think tokens: mean {sum(ok)/len(ok):.0f} over {len(ok)}/{len(test_rows)} parsed')
json.dump(results, open('/content/drive/MyDrive/baseline_heldout.json','w'), indent=2)
print('saved -> baseline_heldout.json')

### What to check before moving to Phase 2

1. **Format held.** Outputs should show `<think>` ... concept named at the end ... `</think>`,
   then a flowing-paragraph solution. If the tags are missing, train more epochs.
2. **Baseline is long.** Expect a few hundred think tokens. If generations are already short,
   the short-form traces have over-fired - lower their share and retrain.
3. **Physics is roughly right.** It need not be perfect; you are measuring token counts, not
   accuracy. But wholesale nonsense means the LoRA has damaged the model.
4. **Save `baseline_heldout.json`.** Phase 3 compares against these exact numbers.